# 📅 2026-06-27 (토) 개발 노트 : score_v6 검증 → Vibe Cluster 백엔드 → 배포 전 체크 완료

## 🎯 오늘의 목표
- [x] score_v6 변별력 검증 (핵심 버그)
- [x] Vibe Cluster Macro 12개 확정 + 백엔드 칩 API
- [x] 배포 전 체크리스트 전부 동작 확인 (05-28 멈춘 OAuth 포함)

## 🛠 진행 상황 및 핵심 기록

**1. score_v6 추천 품질 검증 — 종료**
- 변별력 0.6 → **58.7** (힐링검색 1위 82.0 ~ 꼼찌 23.3). 4개 검색어 실측 전부 통과.
- 의도매칭: Witcher 다크판타지 83.6 vs 힐링 50.4. must_not: 공포 395개 하드제외 ✅.
- breakdown(core/xfactor/gem) + identity + JSON 직렬화 정상.
- 알려진 한계(P2): X-Factor 18이 거의 상수 → 중하위권 점수 부풀림. 단 상위권 무오염(변별은 Core가 함). 나중에 X 만점 18→10.

**2. Vibe Cluster Macro 12개 — 확정**
- 측정 방식 결정: **70점 컷**(정확도 우선). 최고vibe 배정은 99:1 분포 붕괴라 폐기.
- 쓸림 3개 수정: action_thrill(+time_pressure 549→443), challenge_master(재정의 15→160), dark_narrative(narrative→choice 99→56).
- 최종: 12/12 OK, 비율 **7.9:1**(PRD <10:1). 칩 실측 — dark/action/cozy 3칩 9게임 전부 다른 app_id, Witcher dark 1위.
- 파일: `fastapi_app/services/vibe_config.py` (MACRO_VIBES 12개 + get_vibe_list/get_vibe_preferences).

**3. 백엔드 칩 API — 작동**
- `GET /api/v1/games/vibes` (목록), `POST /api/v1/games/recommend/by-vibe?vibe_key=X` (추천, v6 by-preference 재사용).
- recommender 시그니처 확인: `recommend_by_preference(db, preferences, count, must_not, required_tags, use_masking)`, `format_recommendations_by_preference(results, preferences)` — breakdown/identity/strengths 이미 채워짐.

**4. 배포 전 체크 — 전부 실측 종료**
- ✅ 테스트 24/24 PASS (pytest는 컨테이너 미설치 → 임시 설치로 실행. requirements.txt 추가 필요)
- ✅ DB 백업 동작 (39M .sql.gz 실파일, `scripts/backup/backup_db.sh`)
- ✅ 프론트 프로덕션 빌드 통과
- ✅ OAuth 실측 로그인(유저 생성), 토큰 블랙리스트(POST logout 200), CORS(OPTIONS 200)
- ✅ 개인정보처리방침/약관(PIPA 필수항목), 쿠키배너(layout 연결), Umami, 환경변수 키 완비

## 🚨 트러블슈팅 (문제 및 해결)

- **문제 1:** OAuth 로그인 시도 → "소셜 로그인 실패", 8001 기본 화면에 멈춤.
  - **원인:** 코드 버그 아님. 컨테이너 파일시스템 깨짐 (`OSError: [Errno 5] Input/output error: '/app/templates'`). 윈도우+Docker 볼륨 마운트 끊김.
  - **해결:** `docker-compose down && up -d` 완전 재생성 (restart/stop·up으론 안 풀림). → 로그인 정상. **05-28에 적용한 redirect_uri 해결책이 옷았음이 확인됨.**

- **문제 2:** `npm run build` 실패 — `useSearchParams() should be wrapped in a suspense boundary at "/"`.
  - **원인:** page.tsx가 useSearchParams를 최상단에서 사용. Next.js 16은 prerender 시 Suspense 요구. dev엔 안 막히고 build에서만.
  - **해결:** HomePage 본문을 HomeContent로 분리, 바깥을 `<Suspense fallback={GameGridSkeleton}>`로 감쌈. 로직 무변경, 구조만. → 빌드 통과.

- **문제 3:** 옫 Opus가 `from config.database import async_session` 반복 사용(3회).
  - **해결:** `from database import AsyncSessionLocal` + `async with AsyncSessionLocal()`. 옫 Opus 외움.

- **문제 4:** `/api/v1/games/vibes` → `int_parsing app_id=vibes` 에러.
  - **원인:** 라우트 순서. `/{app_id}`(동적)가 `/vibes`(고정)보다 위에 등록 → vibes를 app_id로 파싱.
  - **해결:** `/vibes`를 `/{app_id}` 위로 이동.

## 💡 인사이트 및 다음 할 일

- **배운 점:**
  - **"코드 있음 ≠ 동작함"** — 이 프로젝트 최대 리스크. 오늘 ✅로만 적혀있던 OAuth/백업/빌드/테스트를 전부 실제로 돌려 확인. OAuth는 환경 깨진 것까지 잡음.
  - **추측 금지 → 로그 먼저**가 OAuth를 구함. adapter/settings 추측으로 고쳤으면 멀옩한 코드 망가뜨릴 뻔. `LOGIN_REDIRECT_URL='/'`은 일부러 둔 거(주석에 명시)였음.
  - 커버 73% 미달은 정상(칩=정확도 우선). 커버율 100% 욕심이 분포를 99:1로 깨뜨림.
  - 프로젝트 **A/B 분리 결정**: A=추천 서비스(지금), B=데이터 파이프라인(크롤링·분석·적재, 별도 프로젝트). DB만 공유, 코드 독립. Steam/RAWG/UPSTAGE 키는 .env에 이미 준비됨.

- **다음 할 일:**
  1. **Vibe 칩 프론트** (Project A): GET /vibes fetch → 칩 렌더 → 클릭 by-vibe → GameGrid 표시 → 활성 하이라이트. (백엔드·빌드 다 됨, 화면만)
  2. (사소) requirements.txt에 pytest 추가, pydantic `class Config`→`ConfigDict` deprecation 정리
  3. (별도 세션) Project B 데이터 파이프라인: 수집범위 정의 → Steam ToS → 크롤러 → GPT Batch 60지표 → 임베딩 → 적재. **주의: 새 게임 적재 시 gem_percentile 전체 재계산(모수 변동) → A 점수 스케일 검증이 접점.**
  4. (출시 전) 배포처 결정(Vercel/Railway/자체) → 플랫폼 환경변수 등록 → 백업 cron 서버 적용